In [1]:
# import library 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
#import dataset from .csv file 
dataset=pd.read_csv("Social_Network_Ads.csv")

In [3]:
#our dataset has categorical column so we should convet it into numerical because python can't recognise cate data
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [4]:
dataset=pd.get_dummies(dataset,dtype=int,drop_first=True)

In [5]:
dataset

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,1
1,15810944,35,20000,0,1
2,15668575,26,43000,0,0
3,15603246,27,57000,0,0
4,15804002,19,76000,0,1
...,...,...,...,...,...
395,15691863,46,41000,1,0
396,15706071,51,23000,1,1
397,15654296,50,20000,1,0
398,15755018,36,33000,0,1


In [6]:
# remove user id from dataset because user id no need to prediction
dataset=dataset.drop("User ID",axis=1)

In [7]:
dataset

,Age,EstimatedSalary,Purchased,Gender_Male
0,19,19000,0,1
1,35,20000,0,1
2,26,43000,0,0
3,27,57000,0,0
4,19,76000,0,1
...,...,...,...,...
395,46,41000,1,0
396,51,23000,1,1
397,50,20000,1,0
398,36,33000,0,1


In [8]:
# now our dataset turns 4 coluns 
#next step split input and output 
#before that we need to find how many of them purchased and not purchased according to our dataset
dataset["Purchased"].value_counts()

Purchased
0    257
1    143
Name: count, dtype: int64

In [9]:
dataset.columns

Index(['Age', 'EstimatedSalary', 'Purchased', 'Gender_Male'], dtype='object')

In [10]:
#here we clearly know that Non-purchased (0)-count=257 which means 257 and purchased = 143
#next step split input and output 
independent=dataset[['Age', 'EstimatedSalary', 'Gender_Male']]

In [11]:
dependent=dataset[['Purchased']]

In [12]:
dependent

,Purchased
0,0
1,0
2,0
3,0
4,0
...,...
395,1
396,1
397,1
398,0


In [13]:
#next step split train and test data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(independent, dependent, test_size = 1/3, random_state = 0)

In [14]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [15]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
param_grid = param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'max_features': ['sqrt', 'log2'],
    'criterion': ['gini', 'entropy']}
grid = GridSearchCV( RandomForestClassifier(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_weighted') 
   
# fitting the model for grid search 
Grid_classifier_RandomForestClassifier=grid.fit(X_train, y_train) 

Fitting 5 folds for each of 24 candidates, totalling 120 fits


C:\Anaconda3.12.v\Lib\site-packages\sklearn\base.py:1473: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [16]:
print(grid.best_params_)

{'criterion': 'entropy', 'max_depth': None, 'max_features': 'sqrt', 'n_estimators': 100}


In [17]:
#now pass test dataset to test our model
y_pred=grid.predict(X_test)

In [18]:
#now evalurate our model with answer key (y_test)
# we use eveluation metric is confusion matrix because its classifier problem
from sklearn.metrics import confusion_matrix
cm=confusion_matrix(y_test,y_pred)
print (cm)

[[78  7]
 [ 4 45]]


In [19]:
#confusion matrix shows only classified value by our model
#so we need to find out how well our model perform over this dataset ,
#for that we need to take classification report
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, y_pred)

In [20]:
print(clf_report) #breakdown of our model performance

              precision    recall  f1-score   support

           0       0.95      0.92      0.93        85
           1       0.87      0.92      0.89        49

    accuracy                           0.92       134
   macro avg       0.91      0.92      0.91       134
weighted avg       0.92      0.92      0.92       134



In [21]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test,grid.predict_proba(X_test)[:,1])


0.9633853541416566

In [22]:
# now check our model performance by passing new value on it
age_input=float(input("Age:"))
salary_input=float(input("BMI:"))
sex_male_input=int(input("Sex Male 0 or 1:"))

Age: 30
BMI: 15000
Sex Male 0 or 1: 0


In [23]:
Social_network_Ads_Prediction=grid.predict([[age_input,salary_input,sex_male_input]])
print("Social_network_Ads_Prediction={}".format(Social_network_Ads_Prediction))

Social_network_Ads_Prediction=[1]


In [24]:
import pickle
filename="finalized_model_Random_Forest_Classification.sav"
pickle.dump(grid,open(filename,'wb'))

In [25]:
loaded_model=pickle.load(open("finalized_model_Random_Forest_Classification.sav",'rb'))
result=loaded_model.predict([[age_input,salary_input,sex_male_input]])

In [26]:
result

array([1], dtype=int64)